# GC MBP-1: Development workspace

Start here for notebook work. Read `project_docs/mbp1_research_split_policy.md`. Only Development is available through `scan_development`; it has no holdout override. The cells below verify access and delivery ordering on one Development day. They create no features, labels or models.

Before exploration, write down the research question, eligibility rules, information clock, feature/label horizons and evaluation criteria. Treat the historical ingestion notebook and full-period quality reports as infrastructure evidence; do not use their holdout diagnostics to choose features.

In [ ]:
import sys
from src.data.mbp1_access import scan_development
from src.data.mbp1_splits import POLICY_ID, POLICY_SHA256, SPLITS

development = SPLITS['development']
print('Interpreter:', sys.executable)
print('Policy:', POLICY_ID, POLICY_SHA256)
print('Development:', development.start, 'through', development.end)

## Research contract (complete before analysis)

- Question and falsifiable hypothesis: pending.
- Session/record eligibility and sensitivity checks: pending.
- Information clock, quote validity and source-action scope: pending.
- Lookback, prediction horizon, holding period and boundary purging: pending.
- Development walk-forward folds, costs and success/failure criteria: pending.

Fit transformations only on each training fold. Reset at sessions, observed rolls and split boundaries; discard any feature/label interval crossing them. Validation requires a frozen candidate; Final Test requires the final frozen procedure.

In [ ]:
# Infrastructure smoke check only: one Development session, bounded batches.
day = '2024-09-03'
scanner = scan_development(day, day,
    columns=['session_date_ny', 'event_idx_day', 'instrument_id', 'roll_segment'])
rows = 0
for batch in scanner.to_batches():
    assert all(development.start <= d <= development.end
               for d in batch.column('session_date_ny').unique().to_pylist())
    indices = batch.column('event_idx_day').to_numpy()
    if len(indices):
        assert int(indices[0]) == rows
        assert (indices[1:] == indices[:-1] + 1).all()
    rows += batch.num_rows
print('Development day:', day, 'records:', rows)

## Larger Development selections

`scan_development(columns=[...])` selects the frozen Development range lazily. Iterate `.to_batches()`; do not materialize three years with `.to_table()` or pandas. A date range touching either holdout raises `PermissionError` before file access. Raw bytes, native integer prices and original delivery order remain intact. Split membership alone does not make a session eligible. No trading edge is established.